# tf_lite_development
- Using model V4_E
- Get a sample of snippets of both nothings and chewallows that did well and that failed at each background blend level
- Download model.tflite
- Test the snippets using the tflite model
  - Download them locally
  - develop method to convert them to MFCCs and preprocess them for the model

# Install packages

In [17]:
%pip install scipy
%pip install python_speech_features
%pip install tensorflow
%pip install ai-edge-litert

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 72.6 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ai-edge-litert]m [ai-edge-litert]
Note: you may need to restart the kernel to use updated packages.


# Imports and config

In [20]:
import os
import boto3
import numpy as np
import scipy.io.wavfile
from python_speech_features import mfcc
from numpy import load, savez_compressed
import tensorflow as tf
print("TF version:", tf.__version__)
from ai_edge_litert.interpreter import Interpreter

s3 = boto3.resource('s3')
s3_client = boto3.client("s3")

# Your exact MFCC creation config
N_MFCC           = 23
NFILT             = 32
NFFT              = 512
WINLEN            = 0.01
WINSTEP           = 0.01
EXPERIMENT        = 1
BLANK_TOP_N_ROWS  = 0
TARGET_FRAMES     = 250  # 2.5s at 10ms step

bucket_name = 'hyerman-devbucket'

class_names = ['nothing', 'chewallow'] 

print('done')

TF version: 2.21.0
done


# Define functions

In [28]:
def preprocess_wav_to_model_input(wav_path):
    """
    Enforces (1, 250, 22, 1) regardless of clip length.
    """
    # --- mfcc_creator Lambda ---
    samplerate, samples = scipy.io.wavfile.read(wav_path)
    
    mfcc_feat = mfcc(
        samples, samplerate,
        numcep=N_MFCC,
        nfft=NFFT,
        nfilt=NFILT,
        winlen=WINLEN,
        winstep=WINSTEP
    )
    
    # experiment=1: delete bottom band only
    mfcc_feat = np.delete(mfcc_feat, np.s_[:1], 1)  # → (time_frames, 22)
    
    # --- Enforce exactly TARGET_FRAMES along time axis ---
    n_frames = mfcc_feat.shape[0]
    if n_frames > TARGET_FRAMES:
        mfcc_feat = mfcc_feat[:TARGET_FRAMES, :]
        print(f"  Truncated: {n_frames} → {TARGET_FRAMES} frames")
    elif n_frames < TARGET_FRAMES:
        pad = TARGET_FRAMES - n_frames
        mfcc_feat = np.pad(mfcc_feat, ((0, pad), (0, 0)), mode='constant', constant_values=0)
        print(f"  Padded: {n_frames} → {TARGET_FRAMES} frames")

    # --- Preprocessor Lambda ---
    # Per-sample z-score normalization
    temp_mean = np.mean(mfcc_feat)
    mfcc_feat -= temp_mean
    temp_std = np.std(mfcc_feat)
    mfcc_feat /= temp_std
    
    # Reshape to (1, 250, 22, 1)
    mfcc_input = np.empty((1, mfcc_feat.shape[0], mfcc_feat.shape[1]), dtype=float)
    mfcc_input[0] = mfcc_feat
    mfcc_input = mfcc_input.reshape(
        mfcc_input.shape[0],
        mfcc_input.shape[1],
        mfcc_input.shape[2],
        1
    )
    
    #print(f"  {wav_path} → MFCC shape: {mfcc_input.shape}")
    return mfcc_input.astype(np.float32)


print('done')

done


# Get sample files

In [12]:
select_snippets = {'nothing': {'win': {'020': {'index': 4, 'filename': 'snippets/103302116101414087081_20260210013357_2500_20150_1400_020_unknown.wav', 'prediction': [0.9984005093574524, 0.0015995015855878592], 'prediction_index': 0, 'labels_index': 0}, '200': {'index': 13, 'filename': 'snippets/103302116101414087081_20260210002320_2500_11450_200_200_unknown.wav', 'prediction': [0.9998537302017212, 0.00014630024088546634], 'prediction_index': 0, 'labels_index': 0}, '050': {'index': 14, 'filename': 'snippets/103302116101414087081_20260210004224_2500_9250_500_050_unknown.wav', 'prediction': [0.9999915361404419, 8.462872756354045e-06], 'prediction_index': 0, 'labels_index': 0}, '300': {'index': 16, 'filename': 'snippets/103302116101414087081_20260210005029_2500_27050_800_300_unknown.wav', 'prediction': [0.999847412109375, 0.00015261417138390243], 'prediction_index': 0, 'labels_index': 0}, '030': {'index': 22, 'filename': 'snippets/102444901527529974762_20250602194410_2500_2500_1550_030_nonswallow.wav', 'prediction': [0.9984273910522461, 0.0015725834527984262], 'prediction_index': 0, 'labels_index': 0}, '040': {'index': 31, 'filename': 'snippets/102444901527529974762_20250602192749_2500_2500_1450_040_nonswallow.wav', 'prediction': [0.9997071623802185, 0.0002928427420556545], 'prediction_index': 0, 'labels_index': 0}, '010': {'index': 59, 'filename': 'snippets/103302116101414087081_20260210002018_2500_5425_1675_010_unknown.wav', 'prediction': [1.0, 2.565040801982832e-08], 'prediction_index': 0, 'labels_index': 0}, '400': {'index': 87, 'filename': 'snippets/103302116101414087081_20260210003055_2500_27600_1350_400_unknown.wav', 'prediction': [0.9999998807907104, 1.0973074182629716e-07], 'prediction_index': 0, 'labels_index': 0}, '250': {'index': 92, 'filename': 'snippets/103302116101414087081_20260210003427_2500_26800_550_250_unknown.wav', 'prediction': [0.999981164932251, 1.887556391011458e-05], 'prediction_index': 0, 'labels_index': 0}, '100': {'index': 101, 'filename': 'snippets/103302116101414087081_20260210005230_2500_26600_350_100_unknown.wav', 'prediction': [0.9902666211128235, 0.009733382612466812], 'prediction_index': 0, 'labels_index': 0}, '350': {'index': 174, 'filename': 'snippets/103302116101414087081_20260210001352_2500_22050_800_350_unknown.wav', 'prediction': [0.9984514713287354, 0.001548551837913692], 'prediction_index': 0, 'labels_index': 0}, '150': {'index': 278, 'filename': 'snippets/103302116101414087081_20260210003327_2500_3800_50_150_unknown.wav', 'prediction': [0.9999998807907104, 1.2550613348594197e-07], 'prediction_index': 0, 'labels_index': 0}, '450': {'index': 343, 'filename': 'snippets/103302116101414087081_20260210003427_2500_21800_550_450_unknown.wav', 'prediction': [0.9971086382865906, 0.0028913894202560186], 'prediction_index': 0, 'labels_index': 0}, '500': {'index': 820, 'filename': 'snippets/103302116101414087081_20260210003427_2500_22300_1050_500_unknown.wav', 'prediction': [0.99443119764328, 0.005568807944655418], 'prediction_index': 0, 'labels_index': 0}}, 'loss': {'050': {'index': 12, 'filename': 'snippets/103302116101414087081_20250919231548_2500_7500_200_050_nonswallow.wav', 'prediction': [0.0015545180067420006, 0.998445451259613], 'prediction_index': 1, 'labels_index': 0}, '010': {'index': 998, 'filename': 'snippets/103302116101414087081_20260320012357_2500_10218_1100_010_nonswallow.wav', 'prediction': [0.00890601146966219, 0.9910940527915955], 'prediction_index': 1, 'labels_index': 0}, '100': {'index': 1269, 'filename': 'snippets/103302116101414087081_20260210003852_2500_1700_450_100_unknown.wav', 'prediction': [0.008755682036280632, 0.9912443161010742], 'prediction_index': 1, 'labels_index': 0}, '030': {'index': 5766, 'filename': 'snippets/103302116101414087081_20250809145121_2500_16838_75_030_nonswallow.wav', 'prediction': [0.0003945523058064282, 0.9996054768562317], 'prediction_index': 1, 'labels_index': 0}, '020': {'index': 13132, 'filename': 'snippets/103302116101414087081_20250826151935_2500_18407_0_020_nonswallow.wav', 'prediction': [0.0002776245819404721, 0.9997223019599915], 'prediction_index': 1, 'labels_index': 0}, '040': {'index': 16771, 'filename': 'snippets/103302116101414087081_20260210005633_2500_1525_275_040_unknown.wav', 'prediction': [0.0017058850498870015, 0.9982941746711731], 'prediction_index': 1, 'labels_index': 0}, '400': {'index': 61329, 'filename': 'snippets/103302116101414087081_20250803141020_2500_5000_75_400_nonswallow.wav', 'prediction': [0.003484957618638873, 0.9965150356292725], 'prediction_index': 1, 'labels_index': 0}, '150': {'index': 71515, 'filename': 'snippets/103302116101414087081_20250903020720_2500_12500_100_150_nonswallow.wav', 'prediction': [0.008706954307854176, 0.9912930130958557], 'prediction_index': 1, 'labels_index': 0}, '200': {'index': 155311, 'filename': 'snippets/103302116101414087081_20250826023030_2500_12500_75_200_nonswallow.wav', 'prediction': [0.006335534621030092, 0.9936645030975342], 'prediction_index': 1, 'labels_index': 0}, '500': {'index': 162626, 'filename': 'snippets/103302116101414087081_20250815194659_2500_12367_0_500_nonswallow.wav', 'prediction': [0.005439136177301407, 0.9945608973503113], 'prediction_index': 1, 'labels_index': 0}, '450': {'index': 223653, 'filename': 'snippets/103302116101414087081_20260210002320_2500_5675_1925_450_unknown.wav', 'prediction': [0.00899613555520773, 0.9910038709640503], 'prediction_index': 1, 'labels_index': 0}, '350': {'index': 245850, 'filename': 'snippets/103302116101414087081_20250806194515_2500_2500_50_350_nonswallow.wav', 'prediction': [0.006074682343751192, 0.9939252734184265], 'prediction_index': 1, 'labels_index': 0}}}, 'chewallow': {'win': {'010': {'index': 283505, 'filename': 'snippets/103302116101414087081_20250925150308_2500_21788_-75_010_swallow.wav', 'prediction': [0.003951447084546089, 0.9960485100746155], 'prediction_index': 1, 'labels_index': 1}, '100': {'index': 283522, 'filename': 'snippets/103302116101414087081_20250930151341_2500_4075_400_100_swallow.wav', 'prediction': [0.009789513424038887, 0.9902105331420898], 'prediction_index': 1, 'labels_index': 1}, '050': {'index': 283527, 'filename': 'snippets/103302116101414087081_20251105194219_2500_11277_-300_050_swallow.wav', 'prediction': [0.0007475715246982872, 0.9992523789405823], 'prediction_index': 1, 'labels_index': 1}, '030': {'index': 283530, 'filename': 'snippets/103302116101414087081_20251018200340_2500_18591_-400_030_swallow.wav', 'prediction': [0.0034151768777519464, 0.9965847730636597], 'prediction_index': 1, 'labels_index': 1}, '040': {'index': 283531, 'filename': 'snippets/103302116101414087081_20250822150054_2500_26370_-600_040_swallow.wav', 'prediction': [6.494077751995064e-06, 0.9999935626983643], 'prediction_index': 1, 'labels_index': 1}, '250': {'index': 283568, 'filename': 'snippets/103302116101414087081_20250806012201_2500_6937_-1000_250_swallow.wav', 'prediction': [0.003026387421414256, 0.9969736337661743], 'prediction_index': 1, 'labels_index': 1}, '020': {'index': 283674, 'filename': 'snippets/103302116101414087081_20250930151512_2500_7019_-150_020_swallow.wav', 'prediction': [0.0005934473010711372, 0.9994065761566162], 'prediction_index': 1, 'labels_index': 1}, '300': {'index': 283686, 'filename': 'snippets/103302116101414087081_20250917191236_2500_19653_0_300_nonswallow.wav', 'prediction': [0.0005151772638782859, 0.9994847774505615], 'prediction_index': 1, 'labels_index': 1}, '150': {'index': 283687, 'filename': 'snippets/103302116101414087081_20260317183856_2500_5000_500_150_nonswallow.wav', 'prediction': [0.0011471715988591313, 0.9988528490066528], 'prediction_index': 1, 'labels_index': 1}, '350': {'index': 283705, 'filename': 'snippets/103302116101414087081_20250925015720_2500_11031_-1000_350_swallow.wav', 'prediction': [0.003790640039369464, 0.9962093830108643], 'prediction_index': 1, 'labels_index': 1}, '200': {'index': 283752, 'filename': 'snippets/103302116101414087081_20251021151531_2500_8819_-600_200_swallow.wav', 'prediction': [0.002570956014096737, 0.9974290728569031], 'prediction_index': 1, 'labels_index': 1}, '400': {'index': 284660, 'filename': 'snippets/103302116101414087081_20260315150922_2500_7500_800_400_nonswallow.wav', 'prediction': [0.0010610435856506228, 0.9989389777183533], 'prediction_index': 1, 'labels_index': 1}, '500': {'index': 284678, 'filename': 'snippets/103302116101414087081_20250924145752_2500_5595_-400_500_swallow.wav', 'prediction': [0.009789403527975082, 0.9902106523513794], 'prediction_index': 1, 'labels_index': 1}, '450': {'index': 290240, 'filename': 'snippets/103302116101414087081_20260317145109_2500_16679_800_450_nonswallow.wav', 'prediction': [0.005492922384291887, 0.9945071339607239], 'prediction_index': 1, 'labels_index': 1}}, 'loss': {'020': {'index': 283805, 'filename': 'snippets/103302116101414087081_20250807152855_2500_7178_225_020_swallow.wav', 'prediction': [0.9907537698745728, 0.009246199391782284], 'prediction_index': 0, 'labels_index': 1}, '030': {'index': 284178, 'filename': 'snippets/103302116101414087081_20250802172441_2500_18937_150_030_swallow.wav', 'prediction': [0.998862624168396, 0.0011374370660632849], 'prediction_index': 0, 'labels_index': 1}, '040': {'index': 285179, 'filename': 'snippets/103302116101414087081_20250825150856_2500_19497_150_040_swallow.wav', 'prediction': [0.9921395182609558, 0.00786052830517292], 'prediction_index': 0, 'labels_index': 1}, '050': {'index': 285721, 'filename': 'snippets/103302116101414087081_20250807045843_2500_14682_1000_050_swallow.wav', 'prediction': [0.990167498588562, 0.009832534939050674], 'prediction_index': 0, 'labels_index': 1}, '250': {'index': 285982, 'filename': 'snippets/103302116101414087081_20251018195939_2500_27831_-800_250_swallow.wav', 'prediction': [0.9965818524360657, 0.0034181689843535423], 'prediction_index': 0, 'labels_index': 1}, '010': {'index': 288581, 'filename': 'snippets/103302116101414087081_20251029144803_2500_14676_0_010_swallow.wav', 'prediction': [0.9969366788864136, 0.003063366049900651], 'prediction_index': 0, 'labels_index': 1}, '100': {'index': 289611, 'filename': 'snippets/103302116101414087081_20251207212446_2500_18549_1000_100_swallow.wav', 'prediction': [0.9919959902763367, 0.008004001341760159], 'prediction_index': 0, 'labels_index': 1}, '450': {'index': 295593, 'filename': 'snippets/103302116101414087081_20250902164307_2500_7704_225_450_swallow.wav', 'prediction': [0.9945310950279236, 0.005468985065817833], 'prediction_index': 0, 'labels_index': 1}, '150': {'index': 299596, 'filename': 'snippets/103302116101414087081_20251012142919_2500_14876_-225_150_swallow.wav', 'prediction': [0.9948654770851135, 0.0051345257088541985], 'prediction_index': 0, 'labels_index': 1}, '300': {'index': 308943, 'filename': 'snippets/103302116101414087081_20251018195235_2500_15318_600_300_swallow.wav', 'prediction': [0.9927733540534973, 0.007226637098938227], 'prediction_index': 0, 'labels_index': 1}, '200': {'index': 332330, 'filename': 'snippets/103302116101414087081_20250812184340_2500_23428_800_200_swallow.wav', 'prediction': [0.9933278560638428, 0.006672201212495565], 'prediction_index': 0, 'labels_index': 1}, '400': {'index': 355533, 'filename': 'snippets/103302116101414087081_20251130015755_2500_19690_150_400_swallow.wav', 'prediction': [0.99033522605896, 0.009664732962846756], 'prediction_index': 0, 'labels_index': 1}, '350': {'index': 393599, 'filename': 'snippets/103302116101414087081_20251106155431_2500_23292_150_350_swallow.wav', 'prediction': [0.9930825233459473, 0.006917464546859264], 'prediction_index': 0, 'labels_index': 1}, '500': {'index': 401640, 'filename': 'snippets/103302116101414087081_20250804211458_2500_11894_0_500_nonswallow.wav', 'prediction': [0.9908343553543091, 0.009165655821561813], 'prediction_index': 0, 'labels_index': 1}}}}
# these are select snippets of strong wins and losses of nothing and chewallows at different background blends
# a win is where the prediction equals the label (known value)
# a strong win or loss is where the prediction is <0.01 or >0.99
# a blend is the percent of background noise blended into the sample
#  010 means 1% of the background audio was included in the sample (the model will usually do very well at this blend level)
#  500 means 50% of the background audio was included in the sample (the model is often struggle at this blend level)

# there should be 13 wins and 13 losses for each type (nothing or chewallow)
# in one or two cases there are less

## Download sample files

In [13]:
snip_data_local_dict = {}
local_snip_folder = 'test_snip_files'
# make sure folder exists
os.makedirs(local_snip_folder, exist_ok=True)
os.makedirs(local_snip_folder+'/snippets', exist_ok=True)

for label in select_snippets.keys():
    for success in select_snippets[label].keys():
        for blend in select_snippets[label][success].keys():
            s3_file = select_snippets[label][success][blend]['filename']
            output_file = local_snip_folder + '/' + s3_file
            print(s3_file)
            print(output_file)
            # Download the file
            s3_client.download_file(bucket_name, s3_file, output_file)
            snip_data_local_dict[s3_file] = {'label':label, 'success':success, 'blend':blend, 'local_file_path':output_file}

print(len(snip_data_local_dict))

snippets/103302116101414087081_20260210013357_2500_20150_1400_020_unknown.wav
test_snip_files/snippets/103302116101414087081_20260210013357_2500_20150_1400_020_unknown.wav
snippets/103302116101414087081_20260210002320_2500_11450_200_200_unknown.wav
test_snip_files/snippets/103302116101414087081_20260210002320_2500_11450_200_200_unknown.wav
snippets/103302116101414087081_20260210004224_2500_9250_500_050_unknown.wav
test_snip_files/snippets/103302116101414087081_20260210004224_2500_9250_500_050_unknown.wav
snippets/103302116101414087081_20260210005029_2500_27050_800_300_unknown.wav
test_snip_files/snippets/103302116101414087081_20260210005029_2500_27050_800_300_unknown.wav
snippets/102444901527529974762_20250602194410_2500_2500_1550_030_nonswallow.wav
test_snip_files/snippets/102444901527529974762_20250602194410_2500_2500_1550_030_nonswallow.wav
snippets/102444901527529974762_20250602192749_2500_2500_1450_040_nonswallow.wav
test_snip_files/snippets/102444901527529974762_20250602192749_25

# Get model from S3

In [14]:
my_run_string = "V4_E_chewallow DS2-20260509174952 V4_E_chewallow_DS2/mfcc"
model_name, run_name, datafolder_name = my_run_string.split(" ")

run_job_name = model_name+'_' + datafolder_name+'_' + run_name
run_job_name2 = (model_name+'_'+run_name).replace('_','-')
run_job_name2 = run_job_name2.replace('/','-')
my_s3_output_path = 'model_outputs/'+run_job_name+'/'+ run_job_name2+'/output/output.tar.gz'
print("my_s3_output_path:",my_s3_output_path)

my_s3_output_path: model_outputs/V4_E_chewallow_V4_E_chewallow_DS2/mfcc_DS2-20260509174952/V4-E-chewallow-DS2-20260509174952/output/output.tar.gz


In [15]:
local_folder_name = 'tf_model_folder'

# make sure folder exists
os.makedirs(local_folder_name, exist_ok=True)

tempoutputfile = local_folder_name+'/'+'tempoutputfile.tar.gz'
s3.meta.client.download_file('hyerman-devbucket', my_s3_output_path, tempoutputfile)
print("tempoutputfile",tempoutputfile)
import tarfile
tar = tarfile.open(tempoutputfile)
#print(old_tar.getmembers())
tar.extractall(path=local_folder_name)
print("done")

tempoutputfile tf_model_folder/tempoutputfile.tar.gz
done


## Get tflite model from folder

In [29]:
tflite_path  = 'tf_model_folder/model.tflite'

interpreter = Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("TFLite input shape:", input_details[0]['shape'])
print("TFLite input dtype:", input_details[0]['dtype'])
print("TFLite output shape:", output_details[0]['shape'])
print("TFLite output dtype:", output_details[0]['dtype'])


TFLite input shape: [  1 250  22   1]
TFLite input dtype: <class 'numpy.float32'>
TFLite output shape: [1 2]
TFLite output dtype: <class 'numpy.float32'>


# Run samples through tflite

In [32]:
def predict_tflite(interpreter, x):
    x = x.astype(np.float32)
    interpreter.set_tensor(input_details[0]['index'], x)
    interpreter.invoke()
    return interpreter.get_tensor(output_details[0]['index'])

print("=== TFLite Predictions ===")

# snip_data_local_dict[s3_file] = {'label':label, 'success':success, 'blend':blend, 'local_file_path':output_file}

for snip in snip_data_local_dict.keys():
    label = snip_data_local_dict[snip]['label']
    success = snip_data_local_dict[snip]['success']
    blend = snip_data_local_dict[snip]['blend']
    local_file_path = snip_data_local_dict[snip]['local_file_path']
    
    x    = preprocess_wav_to_model_input(local_file_path)
    pred = predict_tflite(interpreter, x)  # shape (1, 2)
    
    p_class0 = float(pred[0][0])
    p_class1 = float(pred[0][1])
    #p_label    = class_names[1] if p_class1 >= 0.5 else class_names[0]
    #print(f"{local_file_path:30s} → {p_label:12s}  p0={p_class0:.4f}  p1={p_class1:.4f}")
    
    print(label, blend, success, p_class1,'    ',snip)
    # label is what the snippet was originally tagged as by a human (nothing or chewallow)
    # blend is the percent of background noise blended in. (010 is 1%, 100 is 10%)
    # success is whether the original model was very accurate WIN or a horribly inaccurate LOSS.
    # p_class1 is the prediction of tflite of the snippet being a chewallow \
    #    close to zero means the model thinks it is a nothing.
    #    close to one means the model thinks it is a chewallow
    # all of the tflite predictions match the server-side predictions
    #    nothing win all have p_class1 close to zero
    #    nothing loss all have p_class1 close to one
    #    chewallow win all have p_class1 close to one
    #    chewallow loss all have p_class1 close to zero
    # snip is the s3 location of the file

=== TFLite Predictions ===
nothing 020 win 0.0015979664167389274      snippets/103302116101414087081_20260210013357_2500_20150_1400_020_unknown.wav
nothing 200 win 0.0001460722996853292      snippets/103302116101414087081_20260210002320_2500_11450_200_200_unknown.wav
nothing 050 win 8.424012776231393e-06      snippets/103302116101414087081_20260210004224_2500_9250_500_050_unknown.wav
nothing 300 win 0.00015253383025992662      snippets/103302116101414087081_20260210005029_2500_27050_800_300_unknown.wav
nothing 030 win 0.0015708294231444597      snippets/102444901527529974762_20250602194410_2500_2500_1550_030_nonswallow.wav
nothing 040 win 0.000292501732474193      snippets/102444901527529974762_20250602192749_2500_2500_1450_040_nonswallow.wav
nothing 010 win 2.578131841346476e-08      snippets/103302116101414087081_20260210002018_2500_5425_1675_010_unknown.wav
nothing 400 win 1.0973094788369053e-07      snippets/103302116101414087081_20260210003055_2500_27600_1350_400_unknown.wav
nothi